In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import IntegerType, DoubleType
from delta.tables import DeltaTable
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

In [0]:
tables_config = {
    "areas_raw":"area_id",
    "users_raw":"user_id",
    "drivers_raw":"driver_id",
    "vehicles_raw":"vehicle_id",
    "orders_raw":"order_id",
    "deliveries_raw":"delivery_id",
    "payments_raw":"payment_id"
}

In [0]:
def apply_scd2(table_name, pk, df_clean):
    target_table = f"delivery_cata.silver.{table_name}"
    now = current_timestamp()

    meta_cols = {"_ingested_at", "_source_file",
                 "effective_start", "effective_end", "is_current", "_row_hash"}
    biz_cols = [c for c in df_clean.columns if c not in meta_cols]

    window = Window.partitionBy(pk).orderBy(desc("_ingested_at"))
    df_clean = df_clean \
        .withColumn("_rn", row_number().over(window)) \
        .filter(col("_rn") == 1) \
        .drop("_rn")

    df_clean = df_clean \
        .withColumn("effective_start", now) \
        .withColumn("effective_end", lit(None).cast("timestamp")) \
        .withColumn("is_current", lit(True)) \
        .withColumn("_row_hash", sha2(
            concat_ws("||", *[col(c).cast("string") for c in biz_cols]), 256
        ))

    if not spark.catalog.tableExists(target_table):
        df_clean.write.format("delta").saveAsTable(target_table)
        print(f"Created silver.{table_name}: {df_clean.count()} rows")
        return

    target = DeltaTable.forName(spark, target_table)

    target.alias("t").merge(
        df_clean.alias("s"),
        f"t.{pk} = s.{pk} AND t.is_current = true AND t._row_hash != s._row_hash"
    ).whenMatchedUpdate(set={
        "effective_end": "s.effective_start",
        "is_current":    "false"
    }).execute()

    target.alias("t").merge(
        df_clean.alias("s"),
        f"t.{pk} = s.{pk} AND t.is_current = true"
    ).whenNotMatchedInsertAll().execute()

    total   = spark.table(target_table).count()
    current = spark.table(target_table).filter("is_current = true").count()
    print(f"silver.{table_name} → active: {current} | history: {total - current} | total: {total}")

In [0]:
def quarantine(df_bad, table_name, reason):
    """Send bad rows to quarantine table."""
    if df_bad.count() == 0:
        return
    from pyspark.sql.functions import to_json, struct
    df_bad.select(
        lit(table_name).alias("source_table"),
        to_json(struct(*[col(c) for c in df_bad.columns])).alias("raw_data"),
        lit(reason).alias("error_reason"),
        current_timestamp().alias("_ingested_at")
    ).write.format("delta").mode("append") \
     .saveAsTable("delivery_cata.quarantine.bad_rows")
    print(f"{df_bad.count()} rows quarantined → {reason}")

Areas table cleaning 

In [0]:
df = spark.table("delivery_cata.bronze.areas_raw") \
     .withColumn("area_name", trim(col("area_name"))) \
     .withColumn("city", trim(col("city")))

#bad data 
df_bad = df.filter(
    col("area_id").isNull() |                           
    col("area_name").isNull() |                          
    (col("area_name") == "???") | 
    col("city").isNull() |                
    (col("city") == "???") |              
    (col("city").isin("Unknown", "")) |  
    (~col("pincode").rlike("^[0-9]{6}$")) |               
    (col("delivery_charge") < 0)
)

df_good = df.filter(
    col("area_id").isNotNull() &
    col("area_name").isNotNull() &
    (col("area_name") != "???") &
    col("city").isNotNull() &           
    (~col("city").isin("Unknown", "", "???")) & 
    col("pincode").rlike("^[0-9]{6}$") &
    (col("delivery_charge") >= 0)
)
quarantine(df_bad, "areas", "null/invalid area_name, bad pincode, or negative delivery_charge")
apply_scd2("areas", "area_id", df_good)

21970 rows quarantined → null/invalid area_name, bad pincode, or negative delivery_charge
silver.areas → active: 24777 | history: 0 | total: 24777


User table cleaning 

In [0]:

df = spark.table("delivery_cata.bronze.users_raw") \
     .withColumn("name",  trim(col("name"))) \
     .withColumn("email", lower(trim(col("email"))))
     
#bad rows
df_bad = df.filter(
    col("user_id").isNull() |
    col("name").isNull() |
    (length(col("phone")) != 10) |                        
    (~col("email").rlike("^[a-zA-Z0-9._%+\\-]+@[a-zA-Z0-9.\\-]+\\.[a-zA-Z]{2,}$")) |
    col("address").isNull() |                
    (col("address") == "???") |             
    (col("address").isin("Unknown", "")) 
)

df_good = df.filter(
    col("user_id").isNotNull() &
    col("name").isNotNull() &
    (length(col("phone")) == 10) &
    col("email").rlike("^[a-zA-Z0-9._%+\\-]+@[a-zA-Z0-9.\\-]+\\.[a-zA-Z]{2,}$") &
    col("address").isNotNull() &             
    (~col("address").isin("Unknown", "", "???"))
)

quarantine(df_bad, "users", "null user_id/name, invalid phone (not 10 digits), or bad email")
apply_scd2("users", "user_id", df_good)

26440 rows quarantined → null user_id/name, invalid phone (not 10 digits), or bad email
silver.users → active: 23231 | history: 0 | total: 23231


VEHICLES tabel cleaing 

In [0]:
VALID_VEHICLE_TYPES = ["BIKE", "VAN", "TRUCK", "CAR", "SCOOTER"]

df = spark.table("delivery_cata.bronze.vehicles_raw") \
     .withColumn("vehicle_type", upper(trim(coalesce(col("vehicle_type"), lit(""))))) \
     .withColumn("plate_number", trim(col("plate_number"))) \
     .withColumn("status", upper(trim(col("status"))))

df_bad = df.filter(
    col("vehicle_id").isNull() |
    col("vehicle_type").isNull() |
    (~col("vehicle_type").isin(VALID_VEHICLE_TYPES)) |
    (col("plate_number") == "INVALID") |
    col("plate_number").isNull() |
    (col("capacity_kg") <= 0) |                           
    (col("capacity_kg") > 5000)                           
)

df_good = df.filter(
    col("vehicle_id").isNotNull() &
    col("vehicle_type").isNotNull() &
    col("vehicle_type").isin(VALID_VEHICLE_TYPES) &
    col("plate_number").isNotNull() &
    (col("plate_number") != "INVALID") &
    (col("capacity_kg") > 0) &
    (col("capacity_kg") <= 5000)
).withColumn("status",
    when(col("status").isNull(), lit("UNKNOWN")).otherwise(col("status"))
)

quarantine(df_bad, "vehicles", "null/invalid vehicle_type, bad plate_number, or out-of-range capacity_kg")
apply_scd2("vehicles", "vehicle_id", df_good)

15425 rows quarantined → null/invalid vehicle_type, bad plate_number, or out-of-range capacity_kg
silver.vehicles → active: 25926 | history: 0 | total: 25926


DRIVERS table cleaning 

In [0]:
df = spark.table("delivery_cata.bronze.drivers_raw") \
     .withColumn("name", trim(col("name"))) \
     .withColumn("status", upper(trim(col("status")))) \
     .withColumn("vehicle_id", col("vehicle_id").cast(DoubleType())) \
     .withColumn("area_id", col("area_id").cast(DoubleType()))
     
df_bad = df.filter(
    col("driver_id").isNull() |
    col("name").isNull() |
    (col("name") == "???") |
    (col("vehicle_id").isNotNull()) |
    (col("area_id").isNotNull()) |
    (length(col("phone")) != 10)                          
)

df_good = df.filter(
    col("driver_id").isNotNull() &
    col("name").isNotNull() &
    (col("name") != "???") &
    (col("vehicle_id").isNotNull()) &
    (col("area_id").isNotNull()) &
    (length(col("phone")) == 10)
).withColumn("status",
    when(col("status").isNull(), lit("UNKNOWN")).otherwise(col("status"))
)

quarantine(df_bad, "drivers", "null/placeholder name or invalid phone number")
apply_scd2("drivers", "driver_id", df_good)

154265 rows quarantined → null/placeholder name or invalid phone number
silver.drivers → active: 24175 | history: 0 | total: 24175


ORDER table cleaning

In [0]:
VALID_ORDER_STATUS = ["PLACED", "CONFIRMED", "IN_TRANSIT",
                      "DELIVERED", "CANCELLED", "FAILED"]
BAD_LOCATIONS = ["???", "Unknown", "unknown", ""]

df = spark.table("delivery_cata.bronze.orders_raw") \
     .withColumn("order_status", upper(trim(col("order_status")))) \
     .withColumn("pickup_location", trim(col("pickup_location"))) \
     .withColumn("delivery_location", trim(col("delivery_location")))

df_bad = df.filter(
    col("order_id").isNull() |
    col("user_id").isNull() |
    (col("order_amount") <= 0) |                           
    (col("order_amount") >= 999999) |                      
    col("order_status").isNull() |
    (~col("order_status").isin(VALID_ORDER_STATUS)) |
    (col("created_at") > current_timestamp().cast("date")) |
    (col("pickup_location").isNull()) |                
    (col("pickup_location").isin(BAD_LOCATIONS)) |      # Catch pickup placeholders
    (col("delivery_location").isNull()) |
    (col("delivery_location").isin(BAD_LOCATIONS))      # Catch delivery placeholders
)


df_good = df.filter(
    col("order_id").isNotNull() &
    col("user_id").isNotNull() &
    (col("order_amount") > 0) &
    (col("order_amount") < 999999) &
    col("order_status").isNotNull() &
    col("order_status").isin(VALID_ORDER_STATUS) &
    (col("created_at") <= current_timestamp().cast("date")) &
    # Ensure BOTH locations are not null and not placeholders
    (col("pickup_location").isNotNull()) &                
    (~col("pickup_location").isin(BAD_LOCATIONS)) &
    (col("delivery_location").isNotNull()) &                
    (~col("delivery_location").isin(BAD_LOCATIONS))
)

quarantine(df_bad, "orders", "null IDs, negative/outlier amount, invalid status, or future created_at")
apply_scd2("orders", "order_id", df_good)

61390 rows quarantined → null IDs, negative/outlier amount, invalid status, or future created_at
silver.orders → active: 17345 | history: 0 | total: 17345


DELIVERIES table clening 

In [0]:
VALID_DELIVERY_STATUS = ["PENDING", "IN_TRANSIT", "DELIVERED", "FAILED", "CANCELLED"]

df = spark.table("delivery_cata.bronze.deliveries_raw") \
     .withColumn("delivery_status", upper(trim(col("delivery_status"))))

df_bad = df.filter(
    col("delivery_id").isNull() |
    col("order_id").isNull() |
    col("driver_id").isNull() |
    col("delivery_status").isNull() |
    (~col("delivery_status").isin(VALID_DELIVERY_STATUS)) |
    (col("distance_km") <= 0) |
    (col("delivery_time") < col("pickup_time")) |
    (col("vehicle_id").isNull())
)

df_good = df.filter(
    col("delivery_id").isNotNull() &
    col("order_id").isNotNull() &
    col("driver_id").isNotNull() &
    col("delivery_status").isNotNull() &
    col("delivery_status").isin(VALID_DELIVERY_STATUS) &
    (col("distance_km") > 0) &
    (col("delivery_time") >= col("pickup_time")) &
    (col("vehicle_id").isNotNull())
)

quarantine(df_bad, "deliveries", "null IDs, invalid status, negative distance, or delivery_time before pickup_time")
apply_scd2("deliveries", "delivery_id", df_good)

26565 rows quarantined → null IDs, invalid status, negative distance, or delivery_time before pickup_time
silver.deliveries → active: 23891 | history: 0 | total: 23891


PAYMENTS table cleaning

In [0]:
VALID_PAYMENT_METHODS = ["CARD", "UPI", "CASH", "WALLET", "NETBANKING"]
VALID_PAYMENT_STATUS  = ["PENDING", "COMPLETED", "FAILED", "REFUNDED"]

df = spark.table("delivery_cata.bronze.payments_raw") \
     .withColumn("payment_method", upper(trim(col("payment_method")))) \
     .withColumn("payment_status", upper(trim(col("payment_status"))))

df_bad = df.filter(
    col("payment_id").isNull() |
    col("order_id").isNull() |
    (col("amount") <= 0) |                                 
    col("payment_method").isNull() |
    (col("payment_method") == "???") |
    (~col("payment_method").isin(VALID_PAYMENT_METHODS)) |
    col("payment_status").isNull() |
    (~col("payment_status").isin(VALID_PAYMENT_STATUS)) |
    (col("user_id").isNull())
)

df_good = df.filter(
    col("payment_id").isNotNull() &
    col("order_id").isNotNull() &
    (col("amount") > 0) &
    col("payment_method").isNotNull() &
    (col("payment_method") != "???") &
    col("payment_method").isin(VALID_PAYMENT_METHODS) &
    col("payment_status").isNotNull() &
    col("payment_status").isin(VALID_PAYMENT_STATUS) &
    col("user_id").isNotNull()
)

quarantine(df_bad, "payments", "null IDs, negative amount, invalid payment_method or status")
apply_scd2("payments", "payment_id", df_good)

115120 rows quarantined → null IDs, negative amount, invalid payment_method or status
silver.payments → active: 7337 | history: 0 | total: 7337
